# Task 1

In [18]:
import numpy as np
import pygad
import matplotlib.pyplot as plt
import pandas as pd

In [19]:
def read_txt_to_dict(path):

    # dictionary
    slovar = dict()

    # Open the file
    with open(path, 'r') as file:
        
        for line in file:
            
            # Cleaning the line
            splited = line.replace(",", "").replace("[", "").replace("]", "").replace("\n", "").split(" ")
            
            # Filling dictionary
            slovar[splited[0]] = splited[1:]
            
    return slovar


classes = read_txt_to_dict("Data\\Classes.txt")
classrooms = read_txt_to_dict("Data\\Classrooms.txt")
teachers = read_txt_to_dict("Data\\Teachers.txt")

In [ ]:
print(classes)
print(classrooms)
print(teachers)
print()


# Index xy -> x: 0:Programming, 1:Math, 2:Algorithms, 3:OperatingSystems
#          -> y: Which lecture is used

# Predpostavka: Unique classes length is not high
idClasses = []
globalIdClassDictionary = {}
k = 1
slovarTrajanjaUr = {}
for c in classes:
    for i in range(len(classes[c])):
        idClasses.append(k*10 + i)
        slovarTrajanjaUr[k*10+i] = int(classes[c][i])
        # Class name - hourTime - id
        globalIdClassDictionary[k*10 + i] = f"{c}-{k*10 + i} ({classes[c][i]})"
    k+=1
print("idClasses:", idClasses, " -  SlovarTrajanjaUr:", slovarTrajanjaUr)


# Classroom Index: 0:P1, 1:P2, 2:P3, 3:P4
# classroomTime: on index x is start_time and end_time
idClassroom = []
classroomTime = []
n = 0
globalIdClassroomsDictionary = {}

for clas in classrooms:

    c = classrooms[clas]
    classroomTime.append([int(c[0]), int(c[1])])

    globalIdClassroomsDictionary[n] = f"{clas}-{c[0]}-{c[1]}"

    idClassroom.append(n)
    n+=1

print("idClassroom:", idClassroom, " -  classroomTime" ,classroomTime)


# Teacher Index: Anna:0, Brian:1, Charles:2, Diana:3
# teacherClass: on index x are avaible classes that that teacher teachs (ony the first index x)
idTeacher = []
teacherClass = []
globalIdTeachersDictionary = {}
m = 0
for t in teachers:
    idTeacher.append(m)

    globalIdTeachersDictionary[m] = t
    # Which lectures indexed for teachers
    h = []
    for lecture in teachers[t]:
        lec = {"Programming":1, "Math":2, "Algorithms":3, "OperatingSystems":4}

        # Index of a class
        h.append(lec[lecture])

    teacherClass.append(h)
    m+=1


print("idTeacher:", idTeacher, " -  teacherClass:", teacherClass)

print()
print("globalIdClassDictionary", globalIdClassDictionary)
print("globalIdClassroomsDictionary", globalIdClassroomsDictionary)
print("globalIdTeachersDictionary", globalIdTeachersDictionary)

{'Programming': ['2', '3', '2', '2'], 'Math': ['2', '2'], 'Algorithms': ['2', '3'], 'OperatingSystems': ['2', '2']}
{'P1': ['10', '16'], 'P2': ['8', '16'], 'P3': ['10', '16'], 'P4': ['10', '16']}
{'Anna': ['Programming', 'Math'], 'Brian': ['Programming', 'Algorithms', 'OperatingSystems'], 'Charles': ['Math'], 'Diana': ['Algorithms', 'OperatingSystems']}

idClasses: [10, 11, 12, 13, 20, 21, 30, 31, 40, 41]  -  SlovarTrajanjaUr: {10: 2, 11: 3, 12: 2, 13: 2, 20: 2, 21: 2, 30: 2, 31: 3, 40: 2, 41: 2}
idClassroom: [0, 1, 2, 3]  -  classroomTime [[10, 16], [8, 16], [10, 16], [10, 16]]
idTeacher: [0, 1, 2, 3]  -  teacherClass: [[1, 2], [1, 3, 4], [2], [3, 4]]

globalIdClassDictionary {10: 'Programming-10 (2)', 11: 'Programming-11 (3)', 12: 'Programming-12 (2)', 13: 'Programming-13 (2)', 20: 'Math-20 (2)', 21: 'Math-21 (2)', 30: 'Algorithms-30 (2)', 31: 'Algorithms-31 (3)', 40: 'OperatingSystems-40 (2)', 41: 'OperatingSystems-41 (2)'}
globalIdClassroomsDictionary {0: 'P1-10-16', 1: 'P2-8-16', 

In [34]:
# suitable fitness function that will reward schedules that satisfy the conditions and penalize those that do not.
def f(schedule):

    # (idClass, idClassroom, start, idteacher)

    # Global count fitness score
    issue = 0

    # • Every lecture in the classes file must be present in the schedule
    for cid in idClasses:
        find = False    
        
        # Find if exists 
        for lecture in schedule:
            if lecture[0] == cid: 
                find = True
                break
        # If one lecture/class is not found penatly is high
        if find == False: 
            issue += 10


    # Every lecture needs to be assigned to a classroom at a time when that classroom is available. 
    # Filled spots (za vsako učilnico toliko pozicij kot ur)
    spots = [ [ 0 for j in range(razred[1]-razred[0])] for razred in classroomTime]
    

    for lec in schedule:
                
        id_Class, id_Classroom, startHour, id_Teacher = lec
        # You cannot index by float round number
        id_Class, id_Classroom, startHour, id_Teacher = int(id_Class), int(id_Classroom), int(startHour), int(id_Teacher)
    
        # From slovar of hours i get when does this class end 
        endHour = startHour + slovarTrajanjaUr[id_Class]
    
        # What index should it fill (Normalizing so it starts with 0 - for array)
        startIndToFill = startHour - classroomTime[id_Classroom][0]
        endIndToFill = endHour - classroomTime[id_Classroom][0]

        # For how many hours the classroom is open
        dolg = classroomTime[id_Classroom][1] - classroomTime[id_Classroom][0]


        # if the class starts before the classroom opens - penatly (that many hours that is late - increment issue)
        if startHour < classroomTime[id_Classroom][0]:       
            issue += abs(classroomTime[id_Classroom][0] - startIndToFill)
            # The lecture will not be added so +10, because it goes before time 
            issue += 5
            # continue

        # if class goes longer than the classroom is opne (14-17)=14,15,16 -> thats why -1       
        if endHour > classroomTime[id_Classroom][1]:
            issue += (endHour - classroomTime[id_Classroom][1])
            # The lecture will not be added so +10, because it goes over time 
            issue += 5
            # continue
            
        
        validStart = max(startIndToFill, 0)
        validEnd = min(endIndToFill, dolg)
       
        # Fill the spots and check for overlaping 
        # 10-12 -> increments 10, 11
        for i in range(validStart, validEnd):
            spots[id_Classroom][i]+=1

            # Lectures cannot overlap in the same classroom: a single classroom can hold only a single lecture in a given hour.
            # It applies to this also # Teachers cannot teach more than one lecture at the same time.
            # For every value that is more than 1 -> +1 issue

            # PEnantly if more than 1
            if spots[id_Classroom][i] > 1: issue += 3



        # Every lecture needs to be assigned a teacher (That teaches that class) 
        # Which classes can the teacher teach
        canTeach = teacherClass[id_Teacher]
        teachingClass = id_Class // 10

        if teachingClass not in canTeach:
            issue+=10


        
    return issue

# Negating the value to minimize the fitness_score (in this case error rate)
def fitness_function(ga_instance, solution, x_idx):

    # solution is flat so I had to make it 2D
    schedule = [solution[i:i+4] for i in range(0, len(solution), 4)]

    return -f(schedule)




In [79]:

# For each diff class id -> have a prebuilt gene space (idClass, idClassroom, start, idteacher)
# start end will be calculated in fitness if it fits
gene_space = []
for _ in range(len(idClasses)):
    # Extending array further
    gene_space.extend([
        idClasses,
        idClassroom,
        [8, 9, 10, 11, 12, 13, 14, 15],
        idTeacher
    ])


# Genetic Algorithm (sol_per_pop - how many solutions exist in each generation.)
ga_instance = pygad.GA(
    fitness_func=fitness_function, 
    num_generations=300,
    num_parents_mating=10,
    sol_per_pop=35,
    num_genes = len(gene_space),
    gene_space = gene_space,

    # Same teacher for 2 objects, but different class/hour
    allow_duplicate_genes=True)

ga_instance.run()

solution, solution_fitness, solution_idx = ga_instance.best_solution()
print("Best parameter", solution)
print("Fitness", solution_fitness)
print("Index", solution_idx)


Best parameter [13.  1. 14.  1. 31.  0. 13.  1. 20.  0. 10.  0. 12.  3. 10.  0. 30.  2.
 13.  1. 41.  1. 10.  1. 40.  1.  8.  1. 11.  3. 12.  0. 10.  1.  9.  0.
 21.  2. 10.  2.]
Fitness -6
Index 0


In [80]:
def display_schedule(schedule):

    # Group lectures by classroom ID
    classrooms_dict = {}
    for lec in schedule:
        idClass, idClassroom, startHour, idTeacher = lec
        # Convert to int for grouping/indexing safety
        idClass, idClassroom, startHour, idTeacher = int(idClass), int(idClassroom), int(startHour), int(idTeacher)

        # Saving all hours to a specific classroom
        if idClassroom not in classrooms_dict:
            classrooms_dict[idClassroom] = []
        classrooms_dict[idClassroom].append((startHour, idClass, idTeacher))

    print("=" * 60)
    print("DAILY SCHEDULE VISUALIZATION")
    print("=" * 60)

    # Sort classrooms by ID for consistent output
    for idClassroom in sorted(classrooms_dict.keys()):
        
        c_name = globalIdClassroomsDictionary[idClassroom]
        
        print(f"\n📍 {c_name}")
        print("-" * 40)
        
        # Sort lectures in this classroom by start hour
        lectures = sorted(classrooms_dict[idClassroom], key=lambda x: x[0])
        
        for start_hour, id_class, id_teacher in lectures:
            # Resolve names if provided
            cls_name = globalIdClassDictionary[id_class]
            tch_name = globalIdTeachersDictionary[id_teacher]
            
            endHour = start_hour + slovarTrajanjaUr[id_class]
            # Assuming duration can be looked up or displayed if available
            print(f"  [ {start_hour:02d}:00 - {endHour:02d}:00 ] ➔ {cls_name} (Taught by: {tch_name})")
            
    print("\n" + "=" * 60)

schedule = [solution[i:i+4] for i in range(0, len(solution), 4)]
display_schedule(schedule)

DAILY SCHEDULE VISUALIZATION

📍 P1-10-16
----------------------------------------
  [ 10:00 - 12:00 ] ➔ Math-20 (2) (Taught by: Anna)
  [ 13:00 - 16:00 ] ➔ Algorithms-31 (3) (Taught by: Brian)

📍 P2-8-16
----------------------------------------
  [ 08:00 - 10:00 ] ➔ OperatingSystems-40 (2) (Taught by: Brian)
  [ 09:00 - 11:00 ] ➔ Programming-10 (2) (Taught by: Anna)
  [ 10:00 - 12:00 ] ➔ OperatingSystems-41 (2) (Taught by: Brian)
  [ 14:00 - 16:00 ] ➔ Programming-13 (2) (Taught by: Brian)

📍 P3-10-16
----------------------------------------
  [ 10:00 - 12:00 ] ➔ Math-21 (2) (Taught by: Charles)
  [ 13:00 - 15:00 ] ➔ Algorithms-30 (2) (Taught by: Brian)

📍 P4-10-16
----------------------------------------
  [ 10:00 - 12:00 ] ➔ Programming-12 (2) (Taught by: Anna)
  [ 12:00 - 15:00 ] ➔ Programming-11 (3) (Taught by: Anna)

